In [9]:
import os
import json
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import svm
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.under_sampling import RandomUnderSampler
from tensorflow.keras.callbacks import ModelCheckpoint

# Set random seed for NumPy
np.random.seed(42)

# Set random seed for TensorFlow v1
tf.random.set_seed(42)

file_path = 'domain1_train_data.json'
train_data_domain1 = pd.read_json(file_path, lines=True)
file_path = 'domain2_train_data.json'
train_data_domain2 = pd.read_json(file_path, lines=True)
file_path = 'test_data.json'
test_data = pd.read_json(file_path, lines=True)
# split into train and validation data
label_column = 'label'
train_data_domain1, validation_data_domain1 = train_test_split(train_data_domain1, test_size=0.01, random_state=42, stratify=train_data_domain1[label_column])
train_domain1_y = np.array(train_data_domain1[label_column])

## extract train_domain1_x
vectorizer = CountVectorizer()

# Fit the vectorizer to the text data
vectorizer.fit(train_data_domain1['text'].apply(str))

# Transform the text data into a 2D array
train_domain1_x = vectorizer.transform(train_data_domain1['text'].apply(str)).toarray()

validation_domain1_y = np.array(validation_data_domain1[label_column])
validation_domain1_x = vectorizer.transform(validation_data_domain1['text'].apply(str)).toarray()

train_domain2_y = np.array(train_data_domain2[label_column])
train_domain2_x = vectorizer.transform(train_data_domain2['text'].apply(str)).toarray()

# Calculate the number of features
num_features = train_domain1_x.shape[1]

# Create an instance of RandomUnderSampler
undersampler = RandomUnderSampler(random_state=42)

# Resample the majority class
X_resampled, y_resampled = undersampler.fit_resample(train_domain2_x, train_domain2_y)

X_resampled_train, X_resampled_validation, y_resampled_train, y_resampled_validation = train_test_split(X_resampled, y_resampled, test_size=0.05, random_state=42, stratify=y_resampled)

validation_domain2_x = X_resampled_validation
validation_domain2_y = y_resampled_validation

train_x = np.concatenate((train_domain1_x, X_resampled_train ), axis=0)
train_y = np.concatenate((train_domain1_y, y_resampled_train), axis=0)

validation_x = np.concatenate((validation_domain1_x, validation_domain2_x), axis=0)
validation_y = np.concatenate((validation_domain1_y, validation_domain2_y), axis=0)

# Define your model using TensorFlow/Keras
model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(num_features,), kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model_2 = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(num_features,), kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# extract train_domain2_x
model_2.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])

checkpoint_filepath_1 = 'best_model_1.keras'
checkpoint_filepath_2 = 'best_model_2.keras'

# Define separate ModelCheckpoint callbacks for each model
model_checkpoint_callback_1 = ModelCheckpoint(
    filepath=checkpoint_filepath_1,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

model_checkpoint_callback_2 = ModelCheckpoint(
    filepath=checkpoint_filepath_2,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

# Train the first model with the ModelCheckpoint callback
history = model.fit(train_domain1_x, train_domain1_y, epochs=50, batch_size=512,
                    validation_data=(validation_domain1_x, validation_domain1_y),
                    callbacks=[model_checkpoint_callback_1])

# Load the best model saved during training
best_model_1 = models.load_model(checkpoint_filepath_1)

# Train the second model with the new EarlyStopping callback
history_2 = model_2.fit(X_resampled_train, y_resampled_train, epochs=50, batch_size=128,
                        validation_data=(X_resampled_validation, y_resampled_validation),
                        callbacks=[model_checkpoint_callback_2])

# Load the best model saved during training for the second model
best_model_2 = models.load_model(checkpoint_filepath_2)

d:\Anaconda\envs\NLP\Lib\site-packages\keras\src\layers\core\dense.py:86: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5332 - loss: 3.3599
Epoch 1: val_accuracy improved from -inf to 0.66000, saving model to best_model_1.keras
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.5400 - loss: 3.2398 - val_accuracy: 0.6600 - val_loss: 1.6368
Epoch 2/50
 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7298 - loss: 1.4870
Epoch 2: val_accuracy improved from 0.66000 to 0.76000, saving model to best_model_1.keras
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7308 - loss: 1.4716 - val_accuracy: 0.7600 - val_loss: 1.3255
Epoch 3/50
 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7745 - loss: 1.2442
Epoch 3: val_accuracy did not improve from 0.76000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7766 - loss: 1.2371 - val_accuracy: 0.7400 - val_loss: 1.1896
Epoch 4/50
 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8163 - loss: 1.0735
Epoch 4: val_accuracy did not improve from 0.76000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3

In [10]:
# Instantiate Random Forest Classifier
rf_classifier_1 = RandomForestClassifier(
    bootstrap=False,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=500,
    random_state=42
)

# Train the model
rf_classifier_1.fit(train_domain1_x, train_domain1_y)

# Predict the labels of the validation set
validation_predictions_rf = rf_classifier_1.predict(validation_domain1_x)

# Calculate the precision, recall, and F1 score of the model
precision_rf = precision_score(validation_domain1_y, validation_predictions_rf)
recall_rf = recall_score(validation_domain1_y, validation_predictions_rf)
f1_rf = f1_score(validation_domain1_y, validation_predictions_rf)

# Print the precision, recall, and F1 score of the model
print(f'Precision: {precision_rf}')
print(f'Recall: {recall_rf}')
print(f'F1 Score: {f1_rf}')

Precision: 0.7586206896551724
Recall: 0.88
F1 Score: 0.8148148148148148


In [11]:
# Instantiate Random Forest Classifier
rf_classifier_domain_2 = RandomForestClassifier(
    bootstrap=False,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=500,
    random_state=42
)

# Train the model
rf_classifier_domain_2.fit(X_resampled, y_resampled)

# Predict the labels of the validation set
validation_predictions_rf = rf_classifier_domain_2.predict(validation_domain2_x)

# Calculate the precision, recall, and F1 score of the model
precision_rf = precision_score(validation_domain2_y, validation_predictions_rf)
recall_rf = recall_score(validation_domain2_y, validation_predictions_rf)
f1_rf = f1_score(validation_domain2_y, validation_predictions_rf)

# Print the precision, recall, and F1 score of the model
print(f'Precision: {precision_rf}')
print(f'Recall: {recall_rf}')
print(f'F1 Score: {f1_rf}')


Precision: 1.0
Recall: 1.0
F1 Score: 1.0


In [55]:
# Separate minority and majority class instances
minority_class = train_data_domain2[train_data_domain2[label_column] == 1]
majority_class = train_data_domain2[train_data_domain2[label_column] == 0]

# Shuffle the majority class instances
majority_class = majority_class.sample(frac=1, random_state=42)

# Calculate the number of subsets (models) based on the ratio of majority to minority class instances
num_subsets = len(majority_class) // len(minority_class)

# Initialize a list to hold the trained models
models_rf_domain_2 = []

# Loop through each subset, pair minority class with a portion of the majority class, and train a model
for i in range(num_subsets):
    # Select a subset of the majority class
    subset_majority = majority_class[i * len(minority_class):(i + 1) * len(minority_class)]
    
    # Combine minority and majority class instances
    subset_data = pd.concat([minority_class, subset_majority])
    
    # Shuffle the subset data
    subset_data = subset_data.sample(frac=1, random_state=42)
    
    # Split data into features and labels
    subset_x = vectorizer.transform(subset_data['text'].apply(str)).toarray()
    subset_y = np.array(subset_data[label_column])
    
    # Train a classifier (Random Forest in this case)
    rf_classifier = RandomForestClassifier(
        bootstrap=False,
        max_depth=None,
        max_features='sqrt',
        min_samples_leaf=1,
        min_samples_split=5,
        n_estimators=200,
        random_state=42
    )
    rf_classifier.fit(subset_x, subset_y)
    
    # Append trained model to the list
    models_rf_domain_2.append(rf_classifier)


In [13]:
# train an binary classifier to split domain1 and domain2
domain1_y = np.zeros(train_domain1_y.shape)
domain2_y = np.ones(train_domain2_y.shape)
domain_y = np.concatenate((domain1_y, domain2_y), axis=0)
domain_x = np.concatenate((train_domain1_x, train_domain2_x), axis=0)

early_stopping = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

domain_model = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(num_features,)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

domain_model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])

train_domain_x, validation_domain_x, train_domain_y, validation_domain_y = train_test_split(domain_x, domain_y, test_size=0.25, random_state=42, stratify=domain_y)

checkpoint_filepath_domain = 'best_model_domain.keras'

model_checkpoint_callback_domain = ModelCheckpoint(
    filepath=checkpoint_filepath_domain,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

# Train the second model with the new EarlyStopping callback
history_2 = domain_model.fit(train_domain_x, train_domain_y, epochs=20, batch_size=256, validation_data=(validation_domain_x,validation_domain_y), callbacks=[
# Load the best model saved during training for the second model
best_model_2 = models.load_model(checkpoint_filepath_2)



d:\Anaconda\envs\NLP\Lib\site-packages\keras\src\layers\core\dense.py:86: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.9399 - loss: 1.5110 - val_accuracy: 0.9987 - val_loss: 0.4796
Epoch 2/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9988 - loss: 0.3578 - val_accuracy: 1.0000 - val_loss: 0.1167
Epoch 3/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 1.0000 - loss: 0.0860 - val_accuracy: 1.0000 - val_loss: 0.0328
Epoch 4/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 1.0000 - loss: 0.0257 - val_accuracy: 0.9998 - val_loss: 0.0157
Epoch 5/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9997 - loss: 0.0155 - val_accuracy: 0.9998 - val_loss: 0.0120
Epoch 6/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 1.0000 - loss: 0.0099 - val_accuracy: 0.9998 - val_loss: 0.0099
Epoch 7/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 1.0000 - loss: 0.0081 - val_accuracy: 0.9998 - val_loss: 0.0086
Epoch 8/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 1.0000 - loss: 0.0070 - val_accuracy: 0.9998 - v

In [134]:
'''# Use the best parameters to create a new SVM classifier
best_C = 10
best_gamma = 0.01
best_kernel = 'rbf'

best_svm_classifier = svm.SVC(C=best_C, gamma=best_gamma, kernel=best_kernel)

# Train the classifier on your training data
best_svm_classifier.fit(train_domain1_x, train_domain1_y)'''

"# Use the best parameters to create a new SVM classifier\nbest_C = 10\nbest_gamma = 0.01\nbest_kernel = 'rbf'\n\nbest_svm_classifier = svm.SVC(C=best_C, gamma=best_gamma, kernel=best_kernel)\n\n# Train the classifier on your training data\nbest_svm_classifier.fit(train_domain1_x, train_domain1_y)"

In [135]:
'''# Use the best parameters to create a new SVM classifier
best_C = 10
best_gamma = 0.01
best_kernel = 'rbf'

best_svm_classifier = svm.SVC(C=best_C, gamma=best_gamma, kernel=best_kernel)

# Train the classifier on your training data
best_svm_classifier.fit(train_x, train_y)'''
'''accuracy = best_svm_classifier.score(validation_x, validation_y)
print('SVM accuracy:', accuracy)
test_out_put = best_svm_classifier.predict(vectorizer.transform(test_data['text'].apply(str)).toarray())
test_data['label'] = test_out_put
test_data['label'] = test_data['label'].astype(int)
test_data[['id', 'label']].to_csv('sample.csv', index=False)'''

"accuracy = best_svm_classifier.score(validation_x, validation_y)\nprint('SVM accuracy:', accuracy)\ntest_out_put = best_svm_classifier.predict(vectorizer.transform(test_data['text'].apply(str)).toarray())\ntest_data['label'] = test_out_put\ntest_data['label'] = test_data['label'].astype(int)\ntest_data[['id', 'label']].to_csv('sample.csv', index=False)"

In [27]:
model.evaluate(validation_domain1_x, validation_domain1_y)
model_2.evaluate(validation_domain2_x, validation_domain2_y)
# also print f-score
# Assuming y_true contains the true labels and y_pred contains the predicted labels
y_pred = model.predict(validation_domain1_x)
y_pred = np.where(y_pred > 0.5, 1, 0)
y_true = validation_domain1_y
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

# Alternatively, you can directly use the f1_score function
f1 = f1_score(y_true, y_pred)
print('F1 score:', f1)
print('precision:', precision)

y_pred = model_2.predict(validation_domain2_x)
y_pred = np.where(y_pred > 0.5, 1, 0)
y_true = validation_domain2_y
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

# Alternatively, you can directly use the f1_score function
f1 = f1_score(y_true, y_pred)
print('F1 score:', f1)
print('precision:', precision)

# Calculate the accuracy of the classifier on the validation data
'''accuracy = best_svm_classifier.score(validation_domain1_x, validation_domain1_y)
print('SVM accuracy:', accuracy)
accuracy = best_svm_classifier.score(validation_domain2_x, validation_domain2_y)
print('SVM accuracy:', accuracy)
accuracy = best_svm_classifier.score(validation_x, validation_y)
print('SVM accuracy:', accuracy)''' # SVM accuracy: 0.762 SVM accuracy: 0.72 SVM accuracy: 0.7316666666666667

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7937 - loss: 0.6632 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8444 - loss: 0.5734 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
F1 score: 0.8275862068965517
precision: 0.7272727272727273
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
F1 score: 0.8053691275167786
precision: 0.8108108108108109


"accuracy = best_svm_classifier.score(validation_domain1_x, validation_domain1_y)\nprint('SVM accuracy:', accuracy)\naccuracy = best_svm_classifier.score(validation_domain2_x, validation_domain2_y)\nprint('SVM accuracy:', accuracy)\naccuracy = best_svm_classifier.score(validation_x, validation_y)\nprint('SVM accuracy:', accuracy)"

In [28]:
# based on two model, get prediction for test data
test_x = vectorizer.transform(test_data['text'].apply(str)).toarray()
test_y = model.predict(test_x, verbose=0)
test_y_2 = model_2.predict(test_x, verbose=0)
# test_y_4 = best_svm_classifier.predict(test_x)

In [32]:
# split the test data into domain1 and domain2
domain_pred = domain_model.predict(test_x)
domain_pred = np.where(domain_pred > 0.5, 1, 0)
domain1_index = np.where(domain_pred == 0)
domain2_index = np.where(domain_pred == 1)
test_domain_1 = test_data[domain_pred == 0]
test_domain_2 = test_data[domain_pred == 1]

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
domain1: 1995
domain2: 2005


In [53]:
# get the prediction for domain1
test_predict_domain1 = rf_classifier_1.predict(vectorizer.transform(test_domain_1['text'].apply(str)).toarray())
test_predict_domain2 = rf_classifier_domain_2.predict(vectorizer.transform(test_domain_2['text'].apply(str)).toarray())

predictions_per_model_domain2 = []
for rf_model in models_rf_domain_2:
    predictions = rf_model.predict(vectorizer.transform(test_domain_1['text'].apply(str)).toarray())
    predictions_per_model_domain2.append(predictions)
    print('1:', np.sum(predictions == 1))
    print('0:', np.sum(predictions == 0))

# Calculate the average of the predictions
print(test_predict_domain2)

test_predict_domain1 = np.where(test_predict_domain1 > 0.5, 1, 0)
print('1:', np.sum(test_predict_domain1 == 1))
print('0:', np.sum(test_predict_domain1 == 0))
test_predict_domain2 = np.where(test_predict_domain2 > 0.5, 1, 0)
print('1:', np.sum(test_predict_domain2 == 1))
print('0:', np.sum(test_predict_domain2 == 0))

[1 0 1 ... 1 0 1]
1: 1050
0: 945
1: 1164
0: 841


In [54]:
# Initialize the 'label' column with default values
test_data['label'] = 0

# Update the 'label' column with predictions for domain 1
test_data.loc[domain1_index[0], 'label'] = test_predict_domain1.flatten()

# Update the 'label' column with predictions for domain 2
test_data.loc[domain2_index[0], 'label'] = test_predict_domain2.flatten()

# Convert the 'label' column to integer type
test_data['label'] = test_data['label'].astype(int)

print('1:', np.sum(test_data['label'] == 1))
print('0:', np.sum(test_data['label'] == 0))

# Save the predictions to a CSV file
test_data[['id', 'label']].to_csv('sample.csv', index=False)

1: 2214
0: 1786


In [141]:
# binary classification but still in the origianl format
test_y_binary = [1 if i > 0.5 else 0 for i in test_y]
test_y_2_binary = [1 if i > 0.5 else 0 for i in test_y_2]
test_y_2_domain1_binary = [1 if i > 0.5 else 0 for i in test_y_2_domain1]
test_y_3_binary = [1 if i > 0.5 else 0 for i in test_y_3]
test_out_put = test_y_2

In [142]:
# if the two model have different prediction, use the prediction of the model with higher confidence or higher value
test_out_put_binary = [1 if i > 0.5 else 0 for i in test_out_put]
for i in range(len(test_y)):
    if test_out_put_binary[i] != test_y_binary[i]:
        if test_out_put[i] < test_y[i]:
            test_out_put[i] = test_y[i]
            test_out_put_binary[i] = test_y_binary[i]

    if test_out_put_binary[i] != test_y_2_binary[i]:
        if test_out_put[i] < test_y_2[i]:
            test_out_put[i] = test_y_2[i]
            test_out_put_binary[i] = test_y_binary[i]

In [143]:
# copy the prediction to test_prediction
test_prediction = test_out_put.copy()
for i in range(len(test_prediction)):
    test_prediction[i] = 1 if test_prediction[i] > 0.1824 else 0

# count how many 1 and 0 in the prediction
print('1:', np.sum(test_prediction == 1))
print('0:', np.sum(test_prediction == 0))

# save the prediction to the sample.csv in form of have id and label
# make the label into integer
test_data['label'] = test_prediction
test_data['label'] = test_data['label'].astype(int)
test_data[['id', 'label']].to_csv('sample.csv', index=False)

1: 2062
0: 1938
